In [54]:
import pandas as pd

In [55]:
df = pd.read_table('../source_data/PSI_TABLE-hg38.tab')

In [56]:
df_sample = pd.read_table('../source_data/SAMPLE_CONFIG-hg38.tab')

In [57]:
neural = df_sample[(df_sample['Group'] == 'Neural') & (df_sample['Subgroup'] != 'Retina')]['SampleID'].tolist()

In [58]:
others = df_sample[~df_sample['Group'].isin(['Muscle', 'Cell Lines', 'Embryonic Brain', 'Neural'])]['SampleID'].tolist()

In [59]:
muscle =  df_sample[df_sample['Group']=='Muscle']['SampleID'].tolist()

In [60]:
df_select = df.loc[:,df.columns.tolist()[:4]+neural+others+muscle]

In [61]:
df_select['ratio'] = df_select.loc[:,neural].mean(axis=1) / df_select.loc[:,others].mean(axis=1)

/tmp/ipykernel_3800059/2767154390.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_select['ratio'] = df_select.loc[:,neural].mean(axis=1) / df_select.loc[:,others].mean(axis=1)


In [62]:
df_select = df_select[df_select.loc[:,neural].mean(axis=1) > 50]

In [63]:
df_select = df_select[df_select['EVENT'].str.contains('HsaEX')]

In [64]:
df_select = df_select[(df_select['ratio'] > 5) & (df_select['LENGTH'] > 0)]

df_conserve = pd.read_table('../source_data/EVENT_CONSERVATION.tab')

df_conserve = df_conserve[df_conserve['Ass1'] == 'hg38']

df_conserve = df_conserve[df_conserve['ConservedID'] != '\\N']

conserved = []
for event in df_select['EVENT']:
    if len(df_conserve[df_conserve['EventID'] == event]) > 2:
        conserved.append(event)
df_select = df_select[df_select['EVENT'].isin(conserved)]   
df_select.sort_values('ratio', ascending = False).to_csv('../outputs/neuron_up_exon_conserved.csv')

/tmp/ipykernel_3800059/1819897903.py:3: DtypeWarning: Columns (0: Start, 1: End) have mixed types. Specify dtype option on import or set low_memory=False.
  df_conserve = pd.read_table('../source_data/EVENT_CONSERVATION.tab')


In [65]:
df_select = df_select[df_select['LENGTH'] % 3 == 0]
df_select.sort_values('ratio', ascending = False).to_csv('../outputs/neuron_up_inframe_exon_conserved.csv')

In [66]:
df_select[df_select['LENGTH']<=90].to_csv('../outputs/neuron_up_inframe_miniexon_conserved.csv')